In [1]:
import os
import shutil
import random
from PIL import Image
from torchvision import transforms

# ==========================================
# 1. 경로 자동 감지 및 설정
# ==========================================
# 유저 환경에 따라 갈리는 두 가지 바탕화면 경로를 모두 후보로 등록합니다.
PATH_LOCAL = r"C:\Users\user\Desktop\졸작_A모델_학습용_크롭데이터"
PATH_ONEDRIVE = r"C:\Users\user\OneDrive\바탕 화면\졸작_A모델_학습용_크롭데이터"

if os.path.exists(PATH_LOCAL):
    INPUT_DIR = PATH_LOCAL
    OUTPUT_DIR = r"C:\Users\user\Desktop\졸작_최종_파이프라인"
    print("📂 로컬 바탕화면 경로가 감지되었습니다.")
elif os.path.exists(PATH_ONEDRIVE):
    INPUT_DIR = PATH_ONEDRIVE
    OUTPUT_DIR = r"C:\Users\user\OneDrive\바탕 화면\졸작_최종_파이프라인"
    print("☁️ OneDrive 바탕화면 경로가 감지되었습니다.")
else:
    print("❌ [경로 에러] 지정하신 폴더를 찾을 수 없습니다.")
    print("폴더 이름을 다시 확인하시거나, 폴더 주소창을 클릭해 전체 주소를 다시 확인해 주세요.")
    exit()

# 데이터 분할 비율 (80% 학습, 10% 검증, 10% 테스트)
SPLIT_RATIO = {"train": 0.8, "val": 0.1, "test": 0.1}
TARGET_TRAIN_COUNT = 5000 

# ==========================================
# 2. 증식(Augmentation) 파이프라인 세팅
# ==========================================
augment_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1))
])

basic_pipeline = transforms.Compose([
    transforms.Resize((224, 224))
])

# ==========================================
# 3. 파이프라인 실행 로직
# ==========================================
def create_dataset():
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(OUTPUT_DIR, split), exist_ok=True)

    for class_name in os.listdir(INPUT_DIR):
        class_path = os.path.join(INPUT_DIR, class_name)
        if not os.path.isdir(class_path): continue
            
        print(f"\n⚙️ [{class_name}] 파이프라인 처리 중...")
        
        train_save_path = os.path.join(OUTPUT_DIR, "train", class_name)
        val_save_path = os.path.join(OUTPUT_DIR, "val", class_name)
        test_save_path = os.path.join(OUTPUT_DIR, "test", class_name)
        
        if os.path.exists(train_save_path):
            existing_files = len(os.listdir(train_save_path))
            if existing_files >= TARGET_TRAIN_COUNT:
                print(f"  ⏩ 이미 완료된 클래스입니다. (Train {existing_files}장) -> 건너뜁니다.")
                continue
            elif existing_files > 0:
                print(f"  ⚠️ 중단된 흔적 발견! 오염 방지를 위해 [{class_name}] 폴더만 초기화 후 재시작합니다.")
                shutil.rmtree(train_save_path, ignore_errors=True)
                shutil.rmtree(val_save_path, ignore_errors=True)
                shutil.rmtree(test_save_path, ignore_errors=True)
        
        os.makedirs(train_save_path, exist_ok=True)
        os.makedirs(val_save_path, exist_ok=True)
        os.makedirs(test_save_path, exist_ok=True)
            
        files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if len(files) == 0: continue
            
        random.shuffle(files)
        
        total_count = len(files)
        train_idx = int(total_count * SPLIT_RATIO["train"])
        val_idx = train_idx + int(total_count * SPLIT_RATIO["val"])
        
        train_files = files[:train_idx]
        val_files = files[train_idx:val_idx]
        test_files = files[val_idx:]
        
        print(f"  📊 분할 완료: Train({len(train_files)}), Val({len(val_files)}), Test({len(test_files)})")
        
        for f_list, split_name in [(val_files, "val"), (test_files, "test")]:
            for f in f_list:
                try:
                    img = Image.open(os.path.join(class_path, f)).convert("RGB")
                    img = basic_pipeline(img)
                    img.save(os.path.join(OUTPUT_DIR, split_name, class_name, f))
                except: pass
        
        current_train_count = 0
        for f in train_files:
            try:
                img = Image.open(os.path.join(class_path, f)).convert("RGB")
                img_resized = basic_pipeline(img)
                img_resized.save(os.path.join(train_save_path, f"orig_{f}"))
                current_train_count += 1
            except: pass
            
        if current_train_count > 0 and current_train_count < TARGET_TRAIN_COUNT:
            needed_count = TARGET_TRAIN_COUNT - current_train_count
            print(f"  🚀 데이터 불균형 감지: {needed_count}장 증식(Augmentation) 시작!")
            
            aug_count = 0
            fail_count = 0
            
            while current_train_count < TARGET_TRAIN_COUNT:
                random_file = random.choice(train_files)
                try:
                    img = Image.open(os.path.join(class_path, random_file)).convert("RGB")
                    img_aug = augment_pipeline(img)
                    img_aug.save(os.path.join(train_save_path, f"aug_{aug_count}_{random_file}"))
                    
                    current_train_count += 1
                    aug_count += 1
                    fail_count = 0 
                    
                    if current_train_count % 500 == 0:
                        print(f"    ⏳ 열일 중... ({current_train_count} / {TARGET_TRAIN_COUNT}장 완료)")
                        
                except Exception as e:
                    fail_count += 1
                    if fail_count > 100:
                        print(f"  🚨 에러 연속 발생! 파일 손상이 의심되어 다음 폴더로 넘어갑니다.")
                        break
                
        print(f"  💡 최종 Train 세트: {current_train_count}장 구축 완료!")

if __name__ == "__main__":
    print("🚀 [최종 데이터 파이프라인] 가동 시작!")
    create_dataset()
    print("🎉 모든 작업이 완료되었습니다!")

📂 로컬 바탕화면 경로가 감지되었습니다.
🚀 [최종 데이터 파이프라인] 가동 시작!

⚙️ [10_outer_jacket] 파이프라인 처리 중...
  📊 분할 완료: Train(563), Val(70), Test(71)
  🚀 데이터 불균형 감지: 4437장 증식(Augmentation) 시작!
    ⏳ 열일 중... (1000 / 5000장 완료)
    ⏳ 열일 중... (1500 / 5000장 완료)
    ⏳ 열일 중... (2000 / 5000장 완료)
    ⏳ 열일 중... (2500 / 5000장 완료)
    ⏳ 열일 중... (3000 / 5000장 완료)
    ⏳ 열일 중... (3500 / 5000장 완료)
    ⏳ 열일 중... (4000 / 5000장 완료)
    ⏳ 열일 중... (4500 / 5000장 완료)
    ⏳ 열일 중... (5000 / 5000장 완료)
  💡 최종 Train 세트: 5000장 구축 완료!

⚙️ [1_coat] 파이프라인 처리 중...
  📊 분할 완료: Train(511), Val(63), Test(65)
  🚀 데이터 불균형 감지: 4489장 증식(Augmentation) 시작!
    ⏳ 열일 중... (1000 / 5000장 완료)
    ⏳ 열일 중... (1500 / 5000장 완료)
    ⏳ 열일 중... (2000 / 5000장 완료)
    ⏳ 열일 중... (2500 / 5000장 완료)
    ⏳ 열일 중... (3000 / 5000장 완료)
    ⏳ 열일 중... (3500 / 5000장 완료)
    ⏳ 열일 중... (4000 / 5000장 완료)
    ⏳ 열일 중... (4500 / 5000장 완료)
    ⏳ 열일 중... (5000 / 5000장 완료)
  💡 최종 Train 세트: 5000장 구축 완료!

⚙️ [2_padding] 파이프라인 처리 중...
  📊 분할 완료: Train(316), Val(39), Test(40)
  🚀 데이터 불균형 감지: